# 04 · Domain Dominance

Who owns HN's front page? We track which domains (news sites, platforms, personal blogs) dominated each era — and how the ecosystem has shifted from individual blogs to corporate publishing platforms and back to newsletters.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from src.loader import db
from src.nlp import extract_domain
from src.viz import set_style, save

set_style()
con = db()

In [2]:
stories = con.execute("""
    SELECT url, score, YEAR(posted_at) AS year
    FROM stories
    WHERE url IS NOT NULL
      AND score >= 10
      AND YEAR(posted_at) BETWEEN 2008 AND 2024
""").df()

stories['domain'] = stories['url'].apply(extract_domain)
print(f'{len(stories):,} stories with URLs')

647,183 stories with URLs


## Top domains overall

In [3]:
top_domains = (
    stories.groupby('domain')
    .agg(n_stories=('score', 'count'), avg_score=('score', 'mean'))
    .sort_values('n_stories', ascending=False)
    .head(30)
)
display(top_domains)

,n_stories,avg_score
domain,,
github.com,27251,109.509046
nytimes.com,14592,78.226905
techcrunch.com,10258,80.116592
medium.com,9089,71.799978
arstechnica.com,8888,71.590347
twitter.com,8612,107.118672
theguardian.com,8345,68.299700
bloomberg.com,6845,97.273192
youtube.com,6699,68.370802


## Domain share over time (top 10 platforms)

In [4]:
TRACKED = [
    'github.com', 'medium.com', 'nytimes.com', 'techcrunch.com',
    'arstechnica.com', 'substack.com', 'wired.com', 'bloomberg.com',
    'youtube.com', 'reddit.com'
]

total_per_year = stories.groupby('year').size().rename('total')
tracked = stories[stories['domain'].isin(TRACKED)]
pivot = (
    tracked.groupby(['year', 'domain']).size()
    .unstack(fill_value=0)
    .div(total_per_year, axis=0) * 100
)

fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot(ax=ax, linewidth=2, marker='o', markersize=4)
ax.set_title('Domain share of HN stories scoring ≥ 10 (% of year total)', fontsize=14, fontweight='bold')
ax.set_ylabel('Share (%)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
save(fig, '../data/fig_domain_share.png')
plt.show()

## Top 10 domains per year (ranked table)

In [5]:
for year in [2010, 2015, 2018, 2021, 2024]:
    top = (
        stories[stories['year'] == year]
        .groupby('domain').size()
        .sort_values(ascending=False)
        .head(10)
    )
    print(f'\n=== {year} ===')
    print(top.to_string())


=== 2010 ===
domain
techcrunch.com      814
nytimes.com         577
youtube.com         295
wired.com           269
arstechnica.com     257
github.com          249
online.wsj.com      232
readwriteweb.com    169
en.wikipedia.org    163
google.com          133

=== 2015 ===
domain
github.com            2126
medium.com            1298
nytimes.com           1256
techcrunch.com         704
theguardian.com        543
washingtonpost.com     468
bbc.com                436
arstechnica.com        400
wired.com              391
bloomberg.com          387

=== 2018 ===
domain
github.com         1745
nytimes.com        1351
medium.com         1127
bloomberg.com       801
theguardian.com     747
techcrunch.com      614
arstechnica.com     606
youtube.com         435
bbc.com             421
theverge.com        396

=== 2021 ===
domain
github.com          2380
twitter.com         1508
nytimes.com          941
theguardian.com      939
arstechnica.com      760
youtube.com          680
reuters.com     